# Home Credit Default Risk - Exploratory Data Analysis (EDA)

## 1. Problem Context and Overview
Many people struggle to get loans due to insufficient or non-existent credit histories. And, unfortunately, this population is often taken advantage of by untrustworthy lenders.

**Home Credit** strives to broaden financial inclusion for the unbanked population by providing a positive and safe borrowing experience. In order to make sure this underserved population has a positive loan experience, Home Credit makes use of a variety of alternative data--including telco and transactional information--to predict their clients' repayment abilities.

### Goal
Predict the probability (`TARGET`) of whether an applicant will encounter payment difficulties / default on their loan.
- `TARGET = 1`: Client had payment difficulties (late payment more than X days on at least one of the first Y installments)
- `TARGET = 0`: All other cases

In [ ]:
import os
import sys
import warnings
warnings.filterwarnings('ignore')

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns

# Set visual style
sns.set_theme(style='whitegrid', palette='muted')
plt.rcParams['figure.figsize'] = (10, 6)
plt.rcParams['font.size'] = 11

# Add project root to sys.path
sys.path.append(os.path.abspath('..'))
from src.utils import get_data_path, reduce_mem_usage

## 2. Dataset Loading and Initial Inspection

In [ ]:
train_path = get_data_path('application_train.csv')
test_path = get_data_path('application_test.csv')

train_df = pd.read_csv(train_path)
test_df = pd.read_csv(test_path)

print(f"Training set shape: {train_df.shape}")
print(f"Test set shape:     {test_df.shape}")
train_df.head()

## 3. Target Distribution Analysis (Class Imbalance)

In [ ]:
target_counts = train_df['TARGET'].value_counts()
target_pct = train_df['TARGET'].value_counts(normalize=True) * 100

plt.figure(figsize=(8, 5))
bars = plt.bar(['Non-Default (0)', 'Default (1)'], target_counts.values, color=['#2b5c8f', '#d95f02'], width=0.5, edgecolor='black')
for bar, count, pct in zip(bars, target_counts.values, target_pct.values):
    plt.text(bar.get_x() + bar.get_width()/2.0, bar.get_height() + 5000, f"{count:,}\n({pct:.2f}%)", ha='center', fontweight='bold')

plt.title('Target Variable Distribution (Default Imbalance ~8.07%)', fontsize=14, fontweight='bold')
plt.ylabel('Number of Applicants')
plt.ylim(0, max(target_counts.values) * 1.15)
plt.show()

## 4. Missing Values Analysis

In [ ]:
missing = train_df.isnull().sum()
missing_pct = (missing / len(train_df)) * 100
missing_df = pd.DataFrame({'Count': missing, 'Percentage': missing_pct}).query('Count > 0').sort_values('Percentage', ascending=False)

print(f"Total columns with missing values: {len(missing_df)} / {train_df.shape[1]}")

plt.figure(figsize=(12, 8))
sns.barplot(x=missing_df['Percentage'].head(30), y=missing_df.head(30).index, palette='rocket')
plt.title('Top 30 Features with Highest Missing Value Percentage', fontsize=14, fontweight='bold')
plt.xlabel('Missing Percentage (%)')
plt.show()

## 5. Key Feature Analysis: External Sources (`EXT_SOURCE_1`, `EXT_SOURCE_2`, `EXT_SOURCE_3`)

In [ ]:
ext_cols = ['EXT_SOURCE_1', 'EXT_SOURCE_2', 'EXT_SOURCE_3']
fig, axes = plt.subplots(1, 3, figsize=(16, 5))

for idx, col in enumerate(ext_cols):
    sns.kdeplot(train_df.loc[train_df['TARGET'] == 0, col], label='Non-Default (0)', color='#1f77b4', fill=True, alpha=0.3, ax=axes[idx], lw=2)
    sns.kdeplot(train_df.loc[train_df['TARGET'] == 1, col], label='Default (1)', color='#d62728', fill=True, alpha=0.3, ax=axes[idx], lw=2)
    axes[idx].set_title(f'KDE of {col}', fontweight='bold')
    axes[idx].set_xlabel('Normalized Score')
    axes[idx].legend()

plt.suptitle('Strong Separation Observed in External Credit Scores', fontsize=15, fontweight='bold', y=1.03)
plt.tight_layout()
plt.show()

## 6. Demographic Analysis: Age and Employment
- `DAYS_BIRTH`: Recorded as negative number of days relative to loan application.
- `DAYS_EMPLOYED`: Contains the anomalous value `365243` (1000 years in the future, used as placeholder).

In [ ]:
train_df['AGE_YEARS'] = -train_df['DAYS_BIRTH'] / 365.25
train_df['AGE_BIN'] = pd.cut(train_df['AGE_YEARS'], bins=np.arange(20, 75, 5))
age_stats = train_df.groupby('AGE_BIN', observed=True)['TARGET'].agg(['count', 'mean']).reset_index()
age_stats['Default_Rate_%'] = age_stats['mean'] * 100

plt.figure(figsize=(10, 5))
sns.barplot(x='AGE_BIN', y='Default_Rate_%', data=age_stats, palette='Blues_d')
plt.title('Default Rate by Age Group (Higher Risk in Younger Applicants)', fontsize=13, fontweight='bold')
plt.xlabel('Age Group')
plt.ylabel('Default Rate (%)')
plt.xticks(rotation=45)
plt.show()

## 7. Numerical Correlations with Target

In [ ]:
num_cols = ['TARGET', 'EXT_SOURCE_1', 'EXT_SOURCE_2', 'EXT_SOURCE_3', 'AGE_YEARS', 'AMT_INCOME_TOTAL', 'AMT_CREDIT', 'AMT_ANNUITY', 'REGION_RATING_CLIENT']
corr = train_df[num_cols].corr()

plt.figure(figsize=(10, 8))
sns.heatmap(corr, annot=True, fmt='.2f', cmap='coolwarm', center=0, linewidths=0.5)
plt.title('Correlation Heatmap: Key Features and Default Target', fontsize=14, fontweight='bold')
plt.show()